In [1]:
"""
Задание 3
"""
import unicodedata
from dataclasses import dataclass

# Маска для ограничения вычислений 64 битами (имитация uint64)
MASK64 = 0xFFFFFFFFFFFFFFFF


def normalize_word(word: str) -> str:
    """
    Приводим слово к каноническому виду, чтобы:
    - разные представления Unicode (комбинируемые символы) давали одинаковый результат (NFC)
    - сравнение было регистронезависимым (casefold), полезно для естественного языка
    """
    return unicodedata.normalize("NFC", word).casefold()


def fmix64(k: int) -> int:
    """
    Финализатор (bit-mixer) из MurmurHash3: усиливает "лавинный эффект".
    Это важно, если размер таблицы — степень двойки и индекс бакета берётся по низшим битам.
    """
    k &= MASK64
    k ^= (k >> 33) & MASK64
    k = (k * 0xff51afd7ed558ccd) & MASK64
    k ^= (k >> 33) & MASK64
    k = (k * 0xc4ceb9fe1a85ec53) & MASK64
    k ^= (k >> 33) & MASK64
    return k & MASK64


def djb2_plus_normalized(norm_word: str) -> int:
    """
    Модернизированный djb2 (djb2a + финализатор):
    1) norm_word уже нормализован (NFC + casefold)
    2) хешируем байты UTF-8 (поддерживает рус/англ/и т.п.)
    3) используем djb2a: h = h*33 ^ byte
    4) добавляем длину (маленькая "соль" для похожих строк)
    5) финально перемешиваем fmix64
    """
    data = norm_word.encode("utf-8")

    # Классический seed для djb2
    h = 5381
    for b in data:
        # djb2a: умножение на 33 (h*33) и XOR с очередным байтом
        h = ((h * 33) ^ b) & MASK64

    # Учитываем длину (чтобы, например, "a\0" и "a" не вели себя одинаково в некоторых вариантах)
    h ^= len(data)

    # Финальное перемешивание бит
    return fmix64(h)


@dataclass
class HashSetStats:
    # "Коллизии при вставках": сколько элементов уже было в бакете, когда добавляли новое слово.
    # Например, если бакет уже содержит 3 слова, то добавление 4-го увеличит collisions на 3.
    collisions: int = 0
    inserts: int = 0      # сколько уникальных слов реально добавлено
    duplicates: int = 0   # сколько раз пытались добавить уже существующее слово


class HashSetChaining:
    """
    Хеш-множество (словарь наличия слов) через separate chaining:
    - таблица бакетов (списки)
    - при коллизии слова попадают в один и тот же бакет (список)
    """

    def __init__(self, initial_capacity=1024, max_load=0.75):
        # Делаем размер таблицы степенью двойки: удобно считать индекс как hash & (capacity-1)
        cap = 1
        while cap < initial_capacity:
            cap <<= 1

        self.capacity = cap
        self.mask = self.capacity - 1
        self.buckets = [[] for _ in range(self.capacity)]
        self.size = 0
        self.max_load = max_load
        self.stats = HashSetStats()

    def _bucket_index(self, norm_word: str) -> int:
        # Индекс бакета по хешу (берём низшие биты через mask)
        return djb2_plus_normalized(norm_word) & self.mask

    def _rehash(self):
        """
        Увеличиваем таблицу в 2 раза и перераскладываем все слова по новым бакетам.
        Это нужно, чтобы не росло среднее число элементов в бакете (и операции оставались быстрыми).
        """
        old_buckets = self.buckets

        self.capacity <<= 1
        self.mask = self.capacity - 1
        self.buckets = [[] for _ in range(self.capacity)]
        self.size = 0

        for bucket in old_buckets:
            for w in bucket:
                idx = self._bucket_index(w)
                self.buckets[idx].append(w)
                self.size += 1

    def add(self, word: str):
        """
        add <слово>:
        - нормализуем слово
        - при необходимости делаем rehash
        - добавляем в бакет, если слова ещё не было
        - считаем коллизии (сколько элементов уже было в бакете)
        """
        norm = normalize_word(word)

        # Контроль загрузки: если заполнение слишком высокое — расширяем таблицу
        if self.size + 1 > int(self.capacity * self.max_load):
            self._rehash()

        idx = self._bucket_index(norm)
        bucket = self.buckets[idx]

        # Проверяем, не было ли уже такого слова
        for w in bucket:
            if w == norm:
                self.stats.duplicates += 1
                return

        # Новое слово: фиксируем "коллизии при вставке" как длину бакета до вставки
        self.stats.collisions += len(bucket)

        bucket.append(norm)
        self.size += 1
        self.stats.inserts += 1

    def contains(self, word: str) -> bool:
        """
        check <слово>:
        - нормализуем слово
        - идём в соответствующий бакет и ищем его там
        """
        norm = normalize_word(word)
        idx = self._bucket_index(norm)
        return any(w == norm for w in self.buckets[idx])

    def bucket_sizes(self):
        # Размеры бакетов — полезно для анализа распределения
        return [len(b) for b in self.buckets]


# ===== ГОТОВЫЙ ПРИМЕР ВВОДА =====
# Последовательность команд (симулируем ввод пользователя):
commands = [
    "add Привет",
    "add мир",
    "add Мир",          # после normalize_word это то же слово, что и "мир"
    "check привет",
    "check МИР",
    "check python",
    "add python",
    "check python",
]

# ===== ОБРАБОТКА КОМАНД И ВЫВОД =====
hs = HashSetChaining(initial_capacity=16)

print("Команды:")
for cmd in commands:
    print(" ", cmd)

print("\nОтветы на check:")
for line in commands:
    cmd, word = line.split(maxsplit=1)

    if cmd == "add":
        hs.add(word)
    elif cmd == "check":
        print("yes" if hs.contains(word) else "no")

print("\nСтатистика:")
print("  Уникальных слов в словаре:", hs.size)
print("  Коллизии (при вставках):", hs.stats.collisions)
print("  Дубликаты add:", hs.stats.duplicates)
print("  Текущая ёмкость таблицы (бакетов):", hs.capacity)


Команды:
  add Привет
  add мир
  add Мир
  check привет
  check МИР
  check python
  add python
  check python

Ответы на check:
yes
yes
no
yes

Статистика:
  Уникальных слов в словаре: 3
  Коллизии (при вставках): 0
  Дубликаты add: 1
  Текущая ёмкость таблицы (бакетов): 16
